<a href="https://colab.research.google.com/github/tsm-mehmetakiftasoz/tsm_makif/blob/main/1working_progress3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install pandas
!pip install xlsxwriter

In [11]:
import pandas as pd
import numpy as np
from google.colab import files
import os
import zipfile

In [12]:
def temizle_ve_cevir(x):
    try:
        return str(int(float(str(x).strip())))
    except:
        return str(x).strip()

In [13]:
# Temizlenecek değerler listesi
temizlenecekler = ["nan", "Nan", "NaN", ""," ","0","naN","nAn","NAn","nAN"]

def temizle_nan(value):
    if str(value).strip().lower() in [x.lower() for x in temizlenecekler]:
        return pd.NA
    return value


In [14]:
# Güvenli filtreleme fonksiyonu
def dolu_mi(x):
    return pd.notna(x) and str(x).strip() != "" and str(x).strip().upper() != ""


In [ ]:
B3_status = pd.read_excel('/content/B3_status-2026-2022 RU.xlsx')

In [136]:
me2n = pd.read_excel('/content/me2n_2026_2022.xlsx')

FileNotFoundError: [Errno 2] No such file or directory: '/content/me2n_2026_2022.xlsx'

In [ ]:
me5a = pd.read_excel('/content/me5a_2026_2022.xlsx')

In [128]:
import pandas as pd

try:
    # TCMB günlük kur XML adresi
    url = "https://www.tcmb.gov.tr/kurlar/today.xml"
    df = pd.read_xml(url, xpath="//Currency")

    # İhtiyacımız olan dövizlerin kurlarını sözlük olarak alalım (TRY karşılıkları)
    # TCMB kurları genellikle 1, 100 veya 1000 birim üzerinden verilir (Unit sütununa dikkat ederek normalize ediyoruz)
    usd_satiri = df_xml[df_xml['CurrencyCode'] == 'USD'].iloc[0]

    # Değerleri alıp bölme işlemini yapıyor ve 4 basamağa yuvarlıyoruz
    forex_selling = float(usd_satiri['ForexSelling'])
    unit = float(usd_satiri.get('Unit', 1))
    usd_try = round(forex_selling / unit, 4)
    #
    capraz_kur_listesi_tcmb = []
    #
    capraz_kur_listesi_tcmb.append({
        'Çapraz Kur Tipi': f"USD/USD",
        'Değer': 1,
        'Para Birimi': 'USD',
        'Tanım': 'US Dolar',
        #'Kaynak': 'TCMB Doğrudan Veri'
    })
    capraz_kur_listesi_tcmb.append({
        'Çapraz Kur Tipi': f"USD/TRY",
        'Değer': usd_try,
        'Para Birimi': "TRY",
        'Tanım': "Türk Lirası"
        #'Kaynak': 'TCMB Doğrudan Veri'
    })

    for _, row in df.iterrows():
        code = row.get('CurrencyCode')
        isim = row.get('Isim')
        cross_usd = row.get('CrossRateUSD')
        cross_other = row.get('CrossRateOther')

        if code and code != 'TRY':
            # 1. XML'de doğrudan tanımlı USD çapraz kurları varsa ekle
            if pd.notna(cross_usd) and str(cross_usd).strip() != "":
                capraz_kur_listesi_tcmb.append({
                    'Çapraz Kur Tipi': f"{code}/USD",
                    'Değer': float(cross_usd),
                    'Para Birimi': code,
                    'Tanım': isim,
                    #'Kaynak': 'TCMB Doğrudan Veri'
                })

            if pd.notna(cross_other) and str(cross_other).strip() != "":
                capraz_kur_listesi_tcmb.append({
                    'Çapraz Kur Tipi': f"USD/{code}",
                    'Değer': float(cross_other),
                    'Para Birimi': code,
                    'Tanım': isim
                    #'Kaynak': 'TCMB Doğrudan Veri'
                })

    # DataFrame'e dönüştür ve ekrana yazdır

    df_tum_caprazlar = pd.DataFrame(capraz_kur_listesi_tcmb)
    #display(df_tum_caprazlar)


except Exception as e:
    print("Çapraz kurlar listelenirken bir hata oluştu:", e)

,Çapraz Kur Tipi,Değer,Para Birimi,Tanım
0,USD/USD,1.0000,USD,US Dolar
1,USD/TRY,50.2830,TRY,Türk Lirası
2,AUD/USD,1.4719,AUD,AVUSTRALYA DOLARI
3,DKK/USD,6.7603,DKK,DANİMARKA KRONU
4,USD/EUR,1.1058,EUR,EURO
5,USD/GBP,1.3232,GBP,İNGİLİZ STERLİNİ
6,CHF/USD,0.8366,CHF,İSVİÇRE FRANGI
7,SEK/USD,10.0300,SEK,İSVEÇ KRONU
8,CAD/USD,1.4259,CAD,KANADA DOLARI
9,USD/KWD,3.2404,KWD,KUVEYT DİNARI


In [135]:
'''
# 1. Çapraz Kur Tipi sütunundaki benzersiz (unique) değerleri görelim
kurlar_listesi = df_tum_caprazlar['Çapraz Kur Tipi'].unique
print("Mevcut Çapraz Kurlar:")
print(kurlar_listesi)
'''

'\n# 1. Çapraz Kur Tipi sütunundaki benzersiz (unique) değerleri görelim\nkurlar_listesi = df_tum_caprazlar[\'Çapraz Kur Tipi\'].unique\nprint("Mevcut Çapraz Kurlar:")\nprint(kurlar_listesi)\n'

In [130]:
# Tersi alınarak yeni satır olarak eklenecek kur listesi
# (Mevcut Kur, Yeni Eklenecek Kur Tipi, Para Birimi, Tanım formatında)
dondurulecek_kurlar = [
    {'mevcut_tip': 'RUB/USD', 'yeni_tip': 'USD/RUB', 'birim': 'RUB', 'tanim': '1 / (RUB/USD)'},
    {'mevcut_tip': 'USD/TRY', 'yeni_tip': 'TRY/USD', 'birim': 'TRY', 'tanim': '1 / (USD/TRY)'},
    {'mevcut_tip': 'CHF/USD', 'yeni_tip': 'USD/CHF', 'birim': 'CHF', 'tanim': '1 / (CHF/USD)'},
    # İhtiyacınıza göre buraya ekleme yapabilirsiniz
]

yeni_satirlar = []

for kur in dondurulecek_kurlar:
    # İlgili mevcut kur satırını bulalım
    mevcut_row = df_tum_caprazlar[df_tum_caprazlar['Çapraz Kur Tipi'] == kur['mevcut_tip']]

    if not mevcut_row.empty:
        current_val = float(mevcut_row.iloc[0]['Değer'])
        # 0'a bölünme hatasını engellemek için kontrol edelim
        yeni_deger = 1 / current_val if current_val != 0 else np.nan

        yeni_satirlar.append({
            'Çapraz Kur Tipi': kur['yeni_tip'],
            'Değer': yeni_deger,
            'Para Birimi': kur['birim'],
            'Tanım': kur['tanim']
        })

# Yeni satırları topluca DataFrame'e ekliyoruz
if yeni_satirlar:
    df_tum_caprazlar = pd.concat([df_tum_caprazlar, pd.DataFrame(yeni_satirlar)], ignore_index=True)



In [133]:
# Kalması gereken kurlar aşağıya yazılacak.
# Mevcuttaki zplere göre listelenmiş kurlar aşağıda yazılı yenisi gelirse eklenecek.
tutulacak_kurlar = ['USD/USD', 'TRY/USD', 'USD/EUR','USD/RUB','USD/GBP','USD/CHF']  # Örnek liste

# Filtreleme işlemi
gerekli_kurlar = df_tum_caprazlar[df_tum_caprazlar['Çapraz Kur Tipi'].isin(tutulacak_kurlar)].copy()

# Sonucu görüntüleyelim
#print("Filtrelenmiş Çapraz Kurlar:")
#display(gerekli_kurlar)

Filtrelenmiş Çapraz Kurlar:


,Çapraz Kur Tipi,Değer,Para Birimi,Tanım
0,USD/USD,1.000000,USD,US Dolar
4,USD/EUR,1.105800,EUR,EURO
5,USD/GBP,1.323200,GBP,İNGİLİZ STERLİNİ
23,USD/RUB,0.012158,RUB,1 / (RUB/USD)
24,TRY/USD,0.019887,TRY,1 / (USD/TRY)
25,USD/CHF,1.195314,CHF,1 / (CHF/USD)
